In [3]:
# ============================================================
# Notebook 04 : Feature Engineering
# Healthcare Readmission Analytics
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
# ============================================================
# Load Clean Dataset
# ============================================================

df = pd.read_csv("../data/processed/diabetic_data_clean.csv")

print("Dataset loaded successfully.\n")

print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

Dataset loaded successfully.

Rows    : 101766
Columns : 45


In [5]:
# ============================================================
# Dataset Overview
# ============================================================

display(df.head())

print("\nData Types\n")

display(df.dtypes.value_counts())

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,41,0,1,0,0,0,250.83,276,250,1,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,59,0,18,0,0,0,276,250.01,255,9,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,11,5,13,2,0,1,648,250,V27,6,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,44,1,16,0,0,0,8,250.43,403,7,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,51,0,8,0,0,0,197,157,250,5,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO



Data Types



str      32
int64    13
Name: count, dtype: int64

In [6]:
# ============================================================
# Target Variable Encoding
# ============================================================

df["readmitted"] = df["readmitted"].replace({
    "NO": 0,
    "<30": 1,
    ">30": 1
})

print("Target Variable Encoded Successfully\n")

print(df["readmitted"].value_counts())

Target Variable Encoded Successfully

readmitted
0    54864
1    46902
Name: count, dtype: int64


# Target Encoding Observation

## Finding

- The original three-class target variable was converted into a binary classification problem.
- Patients who were readmitted within or after 30 days are grouped into a single **Readmitted** class.

## Business Interpretation

The objective of this project is to predict whether a patient will be readmitted after discharge, regardless of the exact number of days.

## Why It Matters

Binary classification simplifies model development and aligns with a common real-world healthcare decision-making scenario.

In [7]:
# ============================================================
# Age Feature Engineering
# ============================================================

age_mapping = {
    "[0-10)"   : 5,
    "[10-20)"  : 15,
    "[20-30)"  : 25,
    "[30-40)"  : 35,
    "[40-50)"  : 45,
    "[50-60)"  : 55,
    "[60-70)"  : 65,
    "[70-80)"  : 75,
    "[80-90)"  : 85,
    "[90-100)" : 95
}

df["age"] = df["age"].map(age_mapping)

print("Age Feature Engineered Successfully\n")

print(df["age"].head())

Age Feature Engineered Successfully

0     5
1    15
2    25
3    35
4    45
Name: age, dtype: int64


# Age Feature Engineering Observation

## Finding

- The original age ranges were transformed into numerical midpoint values.
- Each age interval is represented by its approximate midpoint (e.g., `[60-70)` → `65`).

## Business Interpretation

Machine learning algorithms require numerical inputs. Converting age groups into midpoint values preserves the natural ordering of age while making the feature suitable for modeling.

## Why It Matters

- Preserves the ordinal relationship between age groups.
- Improves compatibility with machine learning algorithms.
- Enables statistical analysis and feature scaling.

In [8]:
# ============================================================
# Binary Feature Encoding
# ============================================================

binary_columns = [
    "diabetesMed",
    "change"
]

binary_mappings = {
    "Yes": 1,
    "No": 0,
    "Ch": 1
}

for column in binary_columns:
    df[column] = df[column].replace(binary_mappings)

print("Binary Features Encoded Successfully\n")

print(df[binary_columns].head())

Binary Features Encoded Successfully

  diabetesMed change
0           0      0
1           1      1
2           1      0
3           1      1
4           1      1


# Binary Feature Encoding Observation

## Finding

- Binary categorical variables were converted into numerical values.
- Positive responses were encoded as `1`, while negative responses were encoded as `0`.

## Business Interpretation

Binary encoding simplifies categorical information without losing its meaning and prepares these variables for machine learning algorithms.

## Why It Matters

- Reduces preprocessing complexity.
- Makes binary variables directly usable by predictive models.
- Improves computational efficiency.

In [9]:
df["age"].head()

0     5
1    15
2    25
3    35
4    45
Name: age, dtype: int64

In [10]:
df[["diabetesMed", "change"]].head()


,diabetesMed,change
0,0,0
1,1,1
2,1,0
3,1,1
4,1,1


In [11]:
# ============================================================
# Feature Engineering : Total Hospital Visits
# ============================================================

df["total_visits"] = (
    df["number_outpatient"] +
    df["number_emergency"] +
    df["number_inpatient"]
)

print("New Feature Created Successfully.\n")

print(df[[
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "total_visits"
]].head())

New Feature Created Successfully.

   number_outpatient  number_emergency  number_inpatient  total_visits
0                  0                 0                 0             0
1                  0                 0                 0             0
2                  2                 0                 1             3
3                  0                 0                 0             0
4                  0                 0                 0             0


# Total Visits Feature Observation

## Finding

A new feature named **total_visits** was created by combining outpatient, emergency, and inpatient visits.

## Business Interpretation

Instead of analyzing three separate visit-related variables, the total number of hospital visits provides a more comprehensive measure of healthcare utilization.

## Why It Matters

- Represents the patient's overall interaction with the healthcare system.
- May improve prediction of hospital readmission.
- Reduces the complexity of interpreting multiple visit-related variables.

In [12]:
# ============================================================
# Feature Engineering : Total Clinical Procedures
# ============================================================

df["total_procedures"] = (
    df["num_lab_procedures"] +
    df["num_procedures"]
)

print("New Feature Created Successfully.\n")

print(df[[
    "num_lab_procedures",
    "num_procedures",
    "total_procedures"
]].head())

New Feature Created Successfully.

   num_lab_procedures  num_procedures  total_procedures
0                  41               0                41
1                  59               0                59
2                  11               5                16
3                  44               1                45
4                  51               0                51


# Total Procedures Observation

## Finding

A new feature named **total_procedures** was created by combining laboratory procedures and medical procedures.

## Business Interpretation

This feature reflects the overall intensity of medical care received during hospitalization.

## Why It Matters

- Represents overall clinical workload.
- May be associated with disease severity.
- Can improve predictive performance.

In [13]:
# ============================================================
# Feature Engineering : Medication Intensity
# ============================================================

df["medication_intensity"] = pd.cut(
    df["num_medications"],
    bins=[0,10,20,30,100],
    labels=[
        "Low",
        "Moderate",
        "High",
        "Very High"
    ]
)

print("Medication Intensity Created Successfully.\n")

print(df[[
    "num_medications",
    "medication_intensity"
]].head(10))

Medication Intensity Created Successfully.

   num_medications medication_intensity
0                1                  Low
1               18             Moderate
2               13             Moderate
3               16             Moderate
4                8                  Low
5               16             Moderate
6               21                 High
7               12             Moderate
8               28                 High
9               18             Moderate


# Medication Intensity Observation

## Finding

Patients were categorized into four medication intensity groups based on the number of medications prescribed during hospitalization.

## Categories

- Low (≤10)
- Moderate (11–20)
- High (21–30)
- Very High (>30)

## Business Interpretation

Patients receiving more medications often have more complex medical conditions, which may increase the likelihood of hospital readmission.

## Why It Matters

Medication intensity can serve as an indirect indicator of disease severity and treatment complexity.

In [14]:
# ============================================================
# Feature Engineering : Length of Stay Category
# ============================================================

df["stay_category"] = pd.cut(
    df["time_in_hospital"],
    bins=[0,3,7,14],
    labels=[
        "Short",
        "Medium",
        "Long"
    ]
)

print("Stay Category Created Successfully.\n")

print(df[[
    "time_in_hospital",
    "stay_category"
]].head(10))

Stay Category Created Successfully.

   time_in_hospital stay_category
0                 1         Short
1                 3         Short
2                 2         Short
3                 2         Short
4                 1         Short
5                 3         Short
6                 4        Medium
7                 5        Medium
8                13          Long
9                12          Long


# Length of Stay Observation

## Finding

Hospital stay duration was categorized into Short, Medium, and Long stays.

## Business Interpretation

Patients with longer hospital stays generally require more medical attention and may have a higher probability of future readmission.

## Why It Matters

Grouping hospital stay duration simplifies analysis and captures healthcare resource utilization.

In [15]:
# ============================================================
# One-Hot Encoding
# ============================================================

categorical_columns = [
    "race",
    "gender",
    "stay_category",
    "medication_intensity"
]

df = pd.get_dummies(
    df,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

print("One-Hot Encoding Completed Successfully.\n")

print(f"Current Shape : {df.shape}")

One-Hot Encoding Completed Successfully.

Current Shape : (101766, 56)


# One-Hot Encoding Observation

## Finding

Categorical variables were transformed into binary indicator variables using One-Hot Encoding.

## Encoded Features

- Race
- Gender
- Stay Category
- Medication Intensity

## Why One-Hot Encoding?

Machine learning algorithms cannot directly process categorical text values. One-Hot Encoding converts each category into a binary feature while preserving all available information.

Using `drop_first=True` also helps reduce multicollinearity by removing one reference category from each encoded feature.

In [16]:
# ============================================================
# Check Newly Created Features
# ============================================================

encoded_columns = [
    column
    for column in df.columns
    if column.startswith("race_")
    or column.startswith("gender_")
    or column.startswith("stay_category_")
    or column.startswith("medication_intensity_")
]

print("Encoded Features\n")

for column in encoded_columns:
    print(column)

print("\nTotal Encoded Features :", len(encoded_columns))

Encoded Features

race_Asian
race_Caucasian
race_Hispanic
race_Other
gender_Male
gender_Unknown/Invalid
stay_category_Medium
stay_category_Long
medication_intensity_Moderate
medication_intensity_High
medication_intensity_Very High

Total Encoded Features : 11


In [17]:
print(df.shape)

(101766, 56)


In [18]:
encoded_columns

['race_Asian',
 'race_Caucasian',
 'race_Hispanic',
 'race_Other',
 'gender_Male',
 'gender_Unknown/Invalid',
 'stay_category_Medium',
 'stay_category_Long',
 'medication_intensity_Moderate',
 'medication_intensity_High',
 'medication_intensity_Very High']

In [19]:
# ============================================================
# Feature Engineering : Diagnosis Availability
# ============================================================

df["primary_diagnosis_available"] = np.where(
    df["diag_1"].notna(),
    1,
    0
)

print("Primary Diagnosis Feature Created Successfully.\n")

print(df["primary_diagnosis_available"].value_counts())

Primary Diagnosis Feature Created Successfully.

primary_diagnosis_available
1    101766
Name: count, dtype: int64


# Diagnosis Feature Observation

## Finding

A binary feature was created to indicate whether the primary diagnosis is available.

## Business Interpretation

The primary diagnosis is one of the most important clinical variables. Confirming its availability ensures that every patient record contains diagnostic information for downstream analysis.

## Why It Matters

This validation feature helps verify data completeness before machine learning.

In [20]:
# ============================================================
# Standard Scaling
# ============================================================

scaler = StandardScaler()

scale_columns = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "age",
    "total_visits",
    "total_procedures"
]

df[scale_columns] = scaler.fit_transform(df[scale_columns])

print("Feature Scaling Completed Successfully.\n")

df[scale_columns].head()

Feature Scaling Completed Successfully.



,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,age,total_visits,total_procedures
0,-1.137649,-0.106517,-0.785398,-1.848268,-0.291461,-0.21262,-0.503276,-3.321596,-3.824600,-0.524817,-0.173097
1,-0.467653,0.808384,-0.785398,0.243390,-0.291461,-0.21262,-0.503276,0.815784,-3.197277,-0.524817,0.733864
2,-0.802651,-1.631351,2.145781,-0.371804,1.286748,-0.21262,0.288579,-0.735733,-2.569954,0.784215,-1.432764
3,-0.802651,0.045967,-0.199162,-0.002688,-0.291461,-0.21262,-0.503276,-0.218561,-1.942632,-0.524817,0.028450
4,-1.137649,0.401761,-0.785398,-0.986997,-0.291461,-0.21262,-0.503276,-1.252906,-1.315309,-0.524817,0.330770


# Feature Scaling Observation

## Finding

Numerical variables were standardized using StandardScaler.

## Business Interpretation

Scaling ensures that features measured on different scales contribute equally during machine learning model training.

## Why It Matters

- Faster model convergence
- Improved numerical stability
- Better performance for many machine learning algorithms

In [21]:
# ============================================================
# Save Final Engineered Dataset
# ============================================================

df.to_csv(
    "../data/processed/engineered_data.csv",
    index=False
)

print("Engineered dataset saved successfully!")

print(f"Final Dataset Shape : {df.shape}")

Engineered dataset saved successfully!
Final Dataset Shape : (101766, 57)


In [ ]:
# ============================================================
# Final Dataset Quality Check
# ============================================================

print("=" * 50)
print("FINAL DATASET REPORT")
print("=" * 50)

print(f"Rows       : {df.shape[0]}")
print(f"Columns    : {df.shape[1]}")

print("\nMissing Values :", df.isnull().sum().sum())

print("Duplicate Rows :", df.duplicated().sum())

print("\nData Types\n")

print(df.dtypes.value_counts())

FINAL DATASET REPORT
Rows       : 101766
Columns    : 57

Missing Values : 0
Duplicate Rows : 0

Data Types

str        26
int64      17
float64    11
object      3
Name: count, dtype: int64


In [23]:
# ============================================================
# Notebook 04 Completed
# ============================================================

print("=" * 60)
print("NOTEBOOK 04 COMPLETED SUCCESSFULLY")
print("=" * 60)

tasks = [
    "Target Encoding",
    "Age Feature Engineering",
    "Binary Encoding",
    "Total Visits Feature",
    "Total Procedures Feature",
    "Medication Intensity",
    "Stay Category",
    "One-Hot Encoding",
    "Diagnosis Feature",
    "Feature Scaling",
    "Engineered Dataset Saved"
]

for i, task in enumerate(tasks, start=1):
    print(f"{i}. {task}")

print("\nMachine Learning Dataset is Ready.")

NOTEBOOK 04 COMPLETED SUCCESSFULLY
1. Target Encoding
2. Age Feature Engineering
3. Binary Encoding
4. Total Visits Feature
5. Total Procedures Feature
6. Medication Intensity
7. Stay Category
8. One-Hot Encoding
9. Diagnosis Feature
10. Feature Scaling
11. Engineered Dataset Saved

Machine Learning Dataset is Ready.


# Feature Engineering Summary

## Objective

The objective of this notebook was to transform the cleaned healthcare dataset into a machine learning-ready dataset through feature engineering and preprocessing techniques.

---

## Completed Tasks

- Encoded the target variable into binary classes.
- Converted age groups into numerical midpoint values.
- Applied binary encoding to categorical variables.
- Created new features:
  - Total Hospital Visits
  - Total Clinical Procedures
  - Medication Intensity
  - Length of Stay Category
  - Primary Diagnosis Availability
- Applied One-Hot Encoding to categorical features.
- Standardized numerical features using StandardScaler.
- Saved the final engineered dataset for machine learning.

---

## Final Dataset

- Rows: **101,766**
- Features: **(Automatically updated based on the final dataset)**
- Missing Values: **0**
- Duplicate Rows: **0**

---

## Outcome

The dataset is now fully prepared for predictive modeling and will be used in the next notebook to build and evaluate machine learning models for hospital readmission prediction.

# Next Notebook

The next notebook focuses on developing predictive machine learning models using the engineered dataset.

Topics include:

- Logistic Regression
- Decision Tree
- Random Forest
- Model Evaluation
- Confusion Matrix
- ROC Curve
- Feature Importance
- Model Comparison

The objective is to identify the most effective model for predicting patient readmission.